# Datasets Library

The `datasets` library makes it easy to load, explore, and preprocess thousands of public datasets. It uses memory-mapped Arrow files, so even huge datasets don't fill your RAM.

In this notebook we:
1. Load and inspect the IMDB dataset
2. Explore structure: splits, features, statistics
3. Filter, map, and shuffle
4. Visualize class distribution
5. Load a second dataset for comparison

## Setup

In [ ]:
!pip install transformers datasets sentence-transformers pillow -q

## Imports

In [ ]:
from datasets import load_dataset
import matplotlib.pyplot as plt
import collections
import torch

print("Libraries loaded.")

## 1. Load and Inspect the IMDB Dataset

In [ ]:
dataset = load_dataset("imdb")

print("Dataset structure:")
print(dataset)
print(f"\nSplits: {list(dataset.keys())}")
print(f"Train size: {len(dataset['train']):,}")
print(f"Test size : {len(dataset['test']):,}")
print(f"\nFeatures : {dataset['train'].features}")
print(f"\nFirst example:")
print(dataset["train"][0])

## 2. Class Distribution

In [ ]:
label_names = dataset["train"].features["label"].names
print(f"Label names: {label_names}")

for split in ["train", "test"]:
    counts = collections.Counter(dataset[split]["label"])
    print(f"\n{split.capitalize()} set:")
    for label_id, count in sorted(counts.items()):
        print(f"  {label_names[label_id]:10s}: {count:,}")

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
for ax, split in zip(axes, ["train", "test"]):
    counts = collections.Counter(dataset[split]["label"])
    ax.bar([label_names[k] for k in sorted(counts)], [counts[k] for k in sorted(counts)],
           color=["steelblue", "tomato"])
    ax.set_title(f"{split.capitalize()} set")
    ax.set_ylabel("Count")
plt.suptitle("IMDB Label Distribution")
plt.tight_layout()
plt.show()

## 3. Filter, Map, and Shuffle

The three most common dataset operations:

In [ ]:
train = dataset["train"]

# --- Filter: keep only positive reviews ---
positive = train.filter(lambda x: x["label"] == 1)
print(f"All train: {len(train):,}  →  Positive only: {len(positive):,}")

# --- Map: add a 'text_length' column ---
def add_length(example):
    example["text_length"] = len(example["text"].split())
    return example

train_with_len = train.map(add_length)
print(f"\nFirst example text_length: {train_with_len[0]['text_length']} words")

# --- Shuffle and take a small sample ---
small = train_with_len.shuffle(seed=42).select(range(5))
print("\n5 random samples (label, text_length):")
for ex in small:
    print(f"  label={label_names[ex['label']]:10s}  words={ex['text_length']}")

## 4. Review Length Distribution

Understanding text lengths helps choose `max_length` when tokenizing for fine-tuning.

In [ ]:
import numpy as np

# Sample 2000 reviews for speed
sample = train_with_len.shuffle(seed=0).select(range(2000))
lengths = sample["text_length"]

print(f"Min  : {min(lengths)} words")
print(f"Max  : {max(lengths)} words")
print(f"Mean : {np.mean(lengths):.0f} words")
print(f"p50  : {np.percentile(lengths, 50):.0f} words")
print(f"p95  : {np.percentile(lengths, 95):.0f} words")

plt.figure(figsize=(8, 3))
plt.hist(lengths, bins=50, color="steelblue", edgecolor="white")
plt.axvline(512, color="red", linestyle="--", label="512 tokens (BERT max)")
plt.xlabel("Words per review")
plt.ylabel("Count")
plt.title("IMDB Review Length Distribution (2000 samples)")
plt.legend()
plt.tight_layout()
plt.show()